# MoodIn - ST-GCN Graph-Transformer (VA-bridge) | Latih, Uji Performansi, dan Validasi

Pipeline cabang ekspresi MoodIn end-to-end di GPU gratis Kaggle:

`video -> landmark iBUG-68 -> tensor (C=2, T=150, V=68) -> ST-GCN + Transformer -> (valence, arousal) -> mood`

## Perubahan dari versi sebelumnya

| Bagian | Status |
|---|---|
| 1-6 Setup, ekstraksi | dipertahankan, ditambah guard verifikasi |
| 7 Split subject-independent | **BARU** - wajib, split acak membocorkan identitas |
| 8 Latih + cross-validation | diperluas ke 10 fold |
| 9 Inferensi out-of-fold | **BARU** |
| 10 Validasi kategorikal | **BARU** - accuracy, macro-F1, confusion matrix 5x5 |
| 11 Validasi dimensional | **BARU** - CCC, PCC, RMSE, SAGR per dimensi |
| 12 Baseline non-graph | **BARU** - uji falsifikasi klaim GCN |
| 13 Uji performansi runtime | **BARU** - p50/p95 latensi, throughput, memori |
| 14 Ekspor artefak | **BARU** - TorchScript, ONNX, model card |
| 15 Kontrak connector UI | **BARU** - skema IPC + mock publisher |

## Aturan desain yang tidak boleh dilanggar

1. Mood diklasifikasi **sekali saja** di ruang V-A lewat `labels.va_to_mood`. Cabang ST-GCN tidak punya head kategorikal terpisah, supaya tidak konflik dengan STC-rPPG saat fusi.
2. Semua metrik dilaporkan pada split **subject-independent**. Angka dari split acak per-klip tidak boleh masuk proposal.
3. Angka yang belum diukur ditulis `Belum dilakukan`, bukan diisi dari paper lain.

## 1. Cek lingkungan dan GPU

Target deployment akhir adalah **Jetson Orin Nano Super 8GB (JetPack 7.2, CUDA 13.2.1)**, bukan Raspberry Pi. GPU Kaggle hanya untuk pelatihan; angka latensi dari Kaggle tidak boleh dilaporkan sebagai angka Jetson.

In [1]:
import sys, platform
import torch

print('Python      :', sys.version.split()[0])
print('Platform    :', platform.platform())
print('Torch       :', torch.__version__)
print('CUDA avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print('GPU         :', prop.name)
    print('CUDA runtime:', torch.version.cuda)
    print('VRAM (GB)   :', round(prop.total_memory / 1024**3, 2))
else:
    print('PERINGATAN: GPU tidak aktif. Settings -> Accelerator -> GPU T4 x2.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE      :', DEVICE)

Python      : 3.12.13
Platform    : Linux-6.12.90+-x86_64-with-glibc2.35
Torch       : 2.10.0+cu128
CUDA avail  : True
GPU         : Tesla T4
CUDA runtime: 12.8
VRAM (GB)   : 14.56
DEVICE      : cuda


## 2. Dependensi

Aktifkan **Internet** di panel Settings Kaggle sebelum menjalankan sel ini. Versi dikunci agar cocok dengan lingkungan Jetson.

In [2]:
!pip -q install "mediapipe==0.10.14" "protobuf>=4.25.3,<5.0" "opencv-python==4.10.0.84" "numpy>=1.24,<2.0" pyyaml "scikit-learn<1.5.0" onnx onnxruntime

import importlib
for name in ['mediapipe', 'cv2', 'numpy', 'yaml', 'sklearn', 'onnx', 'onnxruntime']:
    try:
        mod = importlib.import_module(name)
        print(name.ljust(14), 'OK  ', getattr(mod, '__version__', '?'))
    except Exception as exc:
        print(name.ljust(14), 'GAGAL', exc)


mediapipe      GAGAL cannot import name 'runtime_version' from 'google.protobuf' (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)
cv2            OK   4.11.0
numpy          OK   1.26.4
yaml           OK   6.0.3
sklearn        OK   1.4.2
onnx           OK   1.22.0
onnxruntime    OK   1.28.0


## 3. Ambil kode repo

- **A. Clone** dari GitHub (butuh Internet aktif).
- **B. Add Data**: unggah repo sebagai Kaggle Dataset lalu salin ke `/kaggle/working`. Pakai ini kalau repo masih privat.

In [3]:
import os, sys

REPO_URL = 'https://github.com/MoodInBME/Software-Architecture.git'
REPO_DIR = '/kaggle/input/datasets/jeremiamanalu/moodin-software/MoodIn-Software'

if not os.path.exists(REPO_DIR):
    rc = os.system('git clone --depth 1 ' + REPO_URL + ' ' + REPO_DIR)
    if rc != 0:
        print('Clone gagal. Pakai Opsi B:')
        print('  !cp -r /kaggle/input/moodin-software-architecture ' + REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

EXPECTED = [
    'models/st_gcn_graph_tran/model.py',
    'models/st_gcn_graph_tran/graph.py',
    'models/st_gcn_graph_tran/train.py',
    'models/st_gcn_graph_tran/config.yaml',
    'data/preprocessing/extract_landmarks_video.py',
    'data/loaders/expression_video_loader.py',
    'evaluation/va_metrics.py',
    'common/labels.py',
]
missing = [p for p in EXPECTED if not os.path.exists(os.path.join(REPO_DIR, p))]
print('Repo   :', REPO_DIR)
print('Lengkap:', not missing)
for p in missing:
    print('  HILANG:', p)
if missing:
    print('Jangan lanjut sebelum semua file ada. Sel berikutnya akan gagal.')

Repo   : /kaggle/input/datasets/jeremiamanalu/moodin-software/MoodIn-Software
Lengkap: False
  HILANG: common/labels.py
Jangan lanjut sebelum semua file ada. Sel berikutnya akan gagal.


## 4. Sanity check arsitektur (data acak)

Uji bentuk tensor dan jalur forward/backward sebelum menyentuh dataset. Kalau gagal di sini, masalahnya di model, bukan di data.

In [4]:
import torch
from models.st_gcn_graph_tran.model import STGCN_GraphTransformer
from models.st_gcn_graph_tran.graph import FaceGraph

N, C, T, V = 4, 2, 150, 68
model = STGCN_GraphTransformer(
    in_channels=C, num_class=5, num_node=V,
    graph_strategy='spatial', max_hop=1, t_kernel=9,
    transformer_layers=2, transformer_heads=8, transformer_ff=512,
    dropout=0.3, head_type='va',
).to(DEVICE)

x = torch.randn(N, C, T, V, device=DEVICE)
out = model(x)

print('Kunci output :', list(out.keys()))
print('Bentuk va    :', tuple(out['va'].shape), '-> harus (N, 2) = valence, arousal')
assert out['va'].shape == (N, 2), 'Head VA salah bentuk'

out['va'].pow(2).mean().backward()
grad_sum = sum(p.grad.abs().sum().item() for p in model.parameters() if p.grad is not None)
print('Total |grad| :', round(grad_sum, 4), '-> harus > 0')
assert grad_sum > 0, 'Gradien nol - graf komputasi terputus'

n_par = sum(p.numel() for p in model.parameters())
print('Parameter    :', format(n_par, ','))
print('Ukuran FP32  :', round(n_par * 4 / 1024**2, 2), 'MB')
print('INT8 estimasi:', round(n_par / 1024**2, 2), 'MB')

graph = FaceGraph(num_node=68, center=33)
print('Adjacency    :', tuple(graph.A.shape), '-> (K, V, V)')
print('SANITY CHECK LULUS')

Kunci output : ['va']
Bentuk va    : (4, 2) -> harus (N, 2) = valence, arousal
Total |grad| : 616.5555 -> harus > 0
Parameter    : 3,066,514
Ukuran FP32  : 11.7 MB
INT8 estimasi: 2.92 MB
Adjacency    : (3, 68, 68) -> (K, V, V)
SANITY CHECK LULUS


## 5. Dataset

| Jalur | Sumber | Kelebihan | Kekurangan |
|---|---|---|---|
| **A. LCK+** | IEEE DataPort, 981 sampel x 68 landmark 2D x 7 emosi | landmark sudah jadi, melewati seluruh tahap MediaPipe | tampaknya sampel statis per-ekspresi, bukan urutan 150 frame |
| **B. CK+ mentah** | mirror Kaggle | urutan frame asli, cocok untuk model temporal | butuh ekstraksi landmark, 20-40 menit |

Jalur A sekaligus jadi **verifikasi numerik `MEDIAPIPE_TO_IBUG68`**: cocokkan landmark hasil MediaPipe Anda dengan landmark LCK+ pada klip yang sama lewat Procrustes. Kalau RMSE ternormalisasi di atas 0.05, peta indeks Anda salah.

> Peringatan: banyak mirror CK+ hanya berisi frame puncak. Model temporal butuh urutan. Struktur yang diharapkan: `root/<emosi>/<klip>/frame_*.png`.

In [5]:
import os, glob

CKPLUS_ROOT = '/kaggle/input/ckplus'          # <-- sesuaikan
PREP_DIR    = '/kaggle/working/prepared/ckplus'
CLIP_LEN    = 150

if os.path.exists(CKPLUS_ROOT):
    print('Isi root :', sorted(os.listdir(CKPLUS_ROOT))[:12])
    imgs = glob.glob(os.path.join(CKPLUS_ROOT, '**', '*.png'), recursive=True)
    imgs += glob.glob(os.path.join(CKPLUS_ROOT, '**', '*.jpg'), recursive=True)
    print('Total gambar :', len(imgs))
    if imgs:
        print('Contoh path  :', imgs[0])
        depth = imgs[0].replace(CKPLUS_ROOT, '').count(os.sep)
        verdict = 'kemungkinan URUTAN' if depth >= 3 else 'kemungkinan FRAME TUNGGAL - cabang temporal akan degenerate'
        print('Kedalaman    :', depth, '->', verdict)
else:
    print('CKPLUS_ROOT tidak ditemukan. Tambahkan lewat Add Data lalu sesuaikan path.')

CKPLUS_ROOT tidak ditemukan. Tambahkan lewat Add Data lalu sesuaikan path.


## 6. Ekstraksi dan cache landmark (sekali saja)

Output: `manifest.json` plus file `.npy` berisi tensor `(2, 150, 68)` ternormalisasi.

In [6]:
!python data/preprocessing/extract_landmarks_video.py \
    --input_root {CKPLUS_ROOT} --layout ckplus \
    --output_dir {PREP_DIR} --clip_len {CLIP_LEN}

Traceback (most recent call last):
  File "/kaggle/input/datasets/jeremiamanalu/moodin-software/MoodIn-Software/data/preprocessing/extract_landmarks_video.py", line 271, in <module>
    main()
  File "/kaggle/input/datasets/jeremiamanalu/moodin-software/MoodIn-Software/data/preprocessing/extract_landmarks_video.py", line 253, in main
    extractor = MediaPipeLandmarkExtractor(static_image_mode=(args.layout == "ckplus"))
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/input/datasets/jeremiamanalu/moodin-software/MoodIn-Software/data/preprocessing/landmark_extraction_mediapipe.py", line 79, in __init__
    import mediapipe as mp  # lazy import
    ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/mediapipe/__init__.py", line 17, in <module>
    import mediapipe.tasks.python as tasks
  File "/usr/local/lib/python3.12/dist-packages/mediapipe/tasks/python/__init__.py", line 17, in <module>
    from . import audio
 

In [7]:
import json, os
from collections import Counter

MANIFEST = os.path.join(PREP_DIR, 'manifest.json')
with open(MANIFEST) as fh:
    manifest = json.load(fh)

entries = manifest['samples'] if isinstance(manifest, dict) and 'samples' in manifest else manifest
print('Jumlah sampel :', len(entries))
print('Kunci entri   :', list(entries[0].keys()))
print('Entri contoh  :', json.dumps(entries[0], indent=2)[:400])
if 'label' in entries[0]:
    print('Distribusi    :', Counter(e['label'] for e in entries))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/prepared/ckplus/manifest.json'

## 7. Split subject-independent

**Ini bagian yang paling sering salah dan paling merusak kredibilitas.**

CK+ memuat 123 subjek, tiap subjek menyumbang beberapa klip. Kalau split dilakukan acak per-klip, klip subjek yang sama muncul di train dan test. Model belajar mengenali *bentuk wajah orang itu*, bukan ekspresinya. Efeknya akurasi terinflasi belasan poin dan runtuh saat dipakai di cermin pada wajah baru - yaitu skenario nyata MoodIn.

ID subjek CK+ ada di path: `S005/001/S005_001_00000011.png` -> subjek `S005`.

In [ ]:
import re, json
import numpy as np
from collections import Counter

SUBJECT_RE = re.compile(r'(S\d{3})')
SUBJECT_KEYS = ('subject', 'subject_id', 'source', 'source_path', 'video', 'path', 'clip', 'npy')

def infer_subject(entry):
    for key in SUBJECT_KEYS:
        val = entry.get(key)
        if isinstance(val, str):
            match = SUBJECT_RE.search(val)
            if match:
                return match.group(1)
    return None

subjects = [infer_subject(e) for e in entries]
n_unknown = sum(s is None for s in subjects)

if n_unknown:
    print('PERINGATAN:', n_unknown, 'dari', len(entries), 'sampel tanpa ID subjek.')
    print('Split subject-independent TIDAK bisa dijamin. Perbaiki manifest dulu:')
    print('tambahkan field "subject" saat ekstraksi landmark.')
    print('Jangan laporkan metrik apa pun sebelum ini beres.')
else:
    counts = Counter(subjects)
    print('Subjek unik :', len(counts))
    print('Klip/subjek : min', min(counts.values()), '| max', max(counts.values()),
          '| rata', round(len(entries) / len(counts), 1))

for i, entry in enumerate(entries):
    entry['subject'] = subjects[i] if subjects[i] is not None else 'UNK' + str(i).zfill(4)

with open(MANIFEST, 'w') as fh:
    json.dump(manifest, fh, indent=2)
print('Manifest diperbarui dengan field "subject".')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold

N_FOLDS = 10
SEED = 42

groups = np.array([e['subject'] for e in entries])
if 'label' in entries[0]:
    y = np.array([e['label'] for e in entries])
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    folds = list(splitter.split(np.zeros(len(entries)), y, groups))
else:
    splitter = GroupKFold(n_splits=N_FOLDS)
    folds = list(splitter.split(np.zeros(len(entries)), groups=groups))

for k, (tr, te) in enumerate(folds):
    overlap = set(groups[tr]) & set(groups[te])
    assert not overlap, 'BOCOR pada fold ' + str(k) + ': ' + str(overlap)
    print('fold', str(k).rjust(2),
          '| train', str(len(tr)).rjust(4), 'klip /', str(len(set(groups[tr]))).rjust(3), 'subjek',
          '| test', str(len(te)).rjust(4), 'klip /', str(len(set(groups[te]))).rjust(3), 'subjek')

FOLD_PATH = '/kaggle/working/folds_subject_independent.json'
fold_records = []
for k, (tr, te) in enumerate(folds):
    fold_records.append(dict(fold=k, train=tr.tolist(), test=te.tolist()))
with open(FOLD_PATH, 'w') as fh:
    json.dump(fold_records, fh)
print()
print('Tidak ada subjek yang bocor antar-split. Fold disimpan:', FOLD_PATH)

## 8. Latih

- **8a** satu fold, untuk iterasi arsitektur cepat.
- **8b** 10 fold penuh, untuk angka yang masuk proposal.

Hiperparameter dari `config.yaml`: 80 epoch, batch 16, lr 1e-3, `DualLoss(alpha=1.0, beta=0.5)` di atas `CCCLoss`.

> Kalau `train.py` belum menerima `--folds` dan `--fold`, tambahkan dua argumen itu dan lewatkan indeks ke `ExpressionLandmarkDataset` sebagai subset. Perubahan sekitar 15 baris dan wajib, karena tanpa itu tidak ada jalan melakukan CV subject-independent.

In [ ]:
!python -m models.st_gcn_graph_tran.train \
    --config models/st_gcn_graph_tran/config.yaml \
    --manifest {MANIFEST} \
    --folds {FOLD_PATH} --fold 0 \
    --out_dir /kaggle/working/checkpoints/fold0

In [ ]:
# 8b. Cross-validation penuh. Perkiraan 5 menit per fold di T4 -> sekitar 50 menit total.
RUN_FULL_CV = False   # <-- set True saat siap

if RUN_FULL_CV:
    for k in range(N_FOLDS):
        print()
        print('=' * 70)
        print('FOLD', k)
        print('=' * 70)
        os.system(
            'python -m models.st_gcn_graph_tran.train'
            ' --config models/st_gcn_graph_tran/config.yaml'
            ' --manifest ' + MANIFEST +
            ' --folds ' + FOLD_PATH +
            ' --fold ' + str(k) +
            ' --out_dir /kaggle/working/checkpoints/fold' + str(k)
        )
else:
    print('RUN_FULL_CV=False -> hanya fold 0 yang dilatih.')
    print('Metrik satu fold TIDAK cukup untuk proposal. Laporkan mean dan std dari 10 fold.')

## 9. Inferensi out-of-fold

Setiap klip diprediksi **hanya oleh model yang tidak pernah melihat subjeknya**. Prediksi out-of-fold digabung, lalu metrik dihitung sekali di atas seluruh dataset. Ini lebih stabil daripada merata-ratakan metrik per fold pada dataset sekecil CK+.

In [ ]:
import numpy as np, torch, json, os, glob
from torch.utils.data import DataLoader, Subset
from models.st_gcn_graph_tran.train import build_model
from data.loaders.expression_video_loader import ExpressionLandmarkDataset
from common.labels import va_to_mood, MOOD_LABELS, NEUTRAL_RADIUS

print('Kelas mood   :', MOOD_LABELS)
print('Radius netral:', NEUTRAL_RADIUS)

full_ds = ExpressionLandmarkDataset(MANIFEST, clip_len=CLIP_LEN)
n = len(full_ds)

pred_va = np.full((n, 2), np.nan, dtype=np.float32)
true_va = np.full((n, 2), np.nan, dtype=np.float32)
true_mood = np.full(n, -1, dtype=np.int64)
fold_of = np.full(n, -1, dtype=np.int64)

available = sorted(glob.glob('/kaggle/working/checkpoints/fold*/stgcn_va_best.pth'))
print('Checkpoint ditemukan:', len(available))
assert available, 'Tidak ada checkpoint. Jalankan sel 8 dulu.'

for ckpt_path in available:
    k = int(ckpt_path.split('fold')[-1].split(os.sep)[0])
    test_idx = folds[k][1]

    ckpt = torch.load(ckpt_path, map_location='cpu')
    net = build_model(ckpt['config'])
    net.load_state_dict(ckpt['model'])
    net.to(DEVICE).eval()

    loader = DataLoader(Subset(full_ds, test_idx.tolist()), batch_size=16, shuffle=False)
    cursor = 0
    with torch.no_grad():
        for xb, vab, moodb in loader:
            va_hat = net(xb.to(DEVICE))['va'].cpu().numpy()
            rows_idx = test_idx[cursor:cursor + len(va_hat)]
            pred_va[rows_idx] = va_hat
            true_va[rows_idx] = vab.numpy()
            true_mood[rows_idx] = moodb.numpy()
            fold_of[rows_idx] = k
            cursor += len(va_hat)
    print('fold', k, 'selesai ->', len(test_idx), 'klip')

covered = fold_of >= 0
print()
print('Cakupan out-of-fold:', int(covered.sum()), '/', n)
if covered.sum() < n:
    print('Sebagian klip belum tercakup karena belum semua fold dilatih.')
    print('Metrik di bawah hanya berlaku untuk subset yang tercakup.')

pred_va_c = pred_va[covered]
true_va_c = true_va[covered]
true_mood_c = true_mood[covered]
pred_mood_c = np.array([va_to_mood(v, a) for v, a in pred_va_c])
if pred_mood_c.ndim > 1:
    pred_mood_c = pred_mood_c[:, 0].astype(np.int64)

np.savez('/kaggle/working/oof_predictions.npz',
         pred_va=pred_va_c, true_va=true_va_c,
         pred_mood=pred_mood_c, true_mood=true_mood_c,
         subject=groups[covered])
print('Prediksi OOF disimpan: /kaggle/working/oof_predictions.npz')

## 10. Uji validasi A - metrik kategorikal (5 kelas mood)

**Accuracy saja tidak cukup.** CK+ sangat tidak seimbang (`contempt` hanya 18 sampel, `happy` 69). Model yang selalu menebak kelas mayoritas bisa terlihat bagus di accuracy. **macro-F1** memberi bobot sama ke setiap kelas dan akan langsung membongkar kegagalan itu.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, classification_report, confusion_matrix)

acc      = accuracy_score(true_mood_c, pred_mood_c)
bal_acc  = balanced_accuracy_score(true_mood_c, pred_mood_c)
macro_f1 = f1_score(true_mood_c, pred_mood_c, average='macro', zero_division=0)
weig_f1  = f1_score(true_mood_c, pred_mood_c, average='weighted', zero_division=0)
kappa    = cohen_kappa_score(true_mood_c, pred_mood_c)

print('Accuracy          :', round(acc, 4))
print('Balanced accuracy :', round(bal_acc, 4))
print('macro-F1          :', round(macro_f1, 4), '  <- angka utama untuk proposal')
print('weighted-F1       :', round(weig_f1, 4))
print('Cohen kappa       :', round(kappa, 4))
print()
gap = acc - macro_f1
if gap > 0.15:
    print('PERINGATAN: selisih accuracy - macroF1 =', round(gap, 3))
    print('Model kemungkinan runtuh ke kelas mayoritas. Periksa confusion matrix.')
print()
print(classification_report(true_mood_c, pred_mood_c,
                           labels=list(range(len(MOOD_LABELS))),
                           target_names=list(MOOD_LABELS), zero_division=0, digits=3))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cm = confusion_matrix(true_mood_c, pred_mood_c, labels=list(range(len(MOOD_LABELS))))
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
panels = [(axes[0], cm, 'Confusion matrix (jumlah)', 'd'),
          (axes[1], cm_norm, 'Confusion matrix (recall per baris)', '.2f')]
for ax, mat, title, fmt in panels:
    im = ax.imshow(mat, cmap='Blues', vmin=0)
    ax.set_xticks(range(len(MOOD_LABELS)))
    ax.set_yticks(range(len(MOOD_LABELS)))
    ax.set_xticklabels(list(MOOD_LABELS), rotation=35, ha='right')
    ax.set_yticklabels(list(MOOD_LABELS))
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Label sebenarnya')
    ax.set_title(title)
    thr = mat.max() / 2.0
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, format(mat[i, j], fmt), ha='center', va='center',
                    color='white' if mat[i, j] > thr else 'black', fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=180)
print('Disimpan: /kaggle/working/confusion_matrix.png')
plt.show()

## 11. Uji validasi B - metrik dimensional (valence dan arousal)

Metrik ini yang benar-benar mengukur cabang ST-GCN, karena head-nya adalah regresi VA. Klasifikasi mood hanyalah kuantisasi di hilir.

| Metrik | Definisi | Kenapa dipakai |
|---|---|---|
| **CCC** | Lin Concordance Correlation Coefficient | standar de-facto ABAW/Aff-Wild2; menghukum bias dan skala sekaligus |
| **PCC** | Pearson | hanya bentuk korelasi; **tidak cukup sendirian** karena buta terhadap offset konstan |
| **RMSE** | root mean squared error | besaran galat dalam satuan VA |
| **SAGR** | Sign Agreement Rate | apakah tanda positif/negatif benar - relevan langsung untuk keputusan intervensi |

Laporkan CCC valence dan CCC arousal **terpisah**. Merata-ratakannya menyembunyikan fakta umum bahwa arousal jauh lebih sulit diprediksi dari ekspresi wajah saja - dan justru itulah alasan cabang rPPG ada.

In [ ]:
from evaluation.va_metrics import evaluate_va

def ccc(y_true, y_pred):
    mt, mp = y_true.mean(), y_pred.mean()
    vt, vp = y_true.var(), y_pred.var()
    cov = ((y_true - mt) * (y_pred - mp)).mean()
    denom = vt + vp + (mt - mp) ** 2
    return float(2 * cov / denom) if denom > 0 else 0.0

def pcc(y_true, y_pred):
    if y_true.std() == 0 or y_pred.std() == 0:
        return 0.0
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def rmse(y_true, y_pred):
    return float(np.sqrt(((y_true - y_pred) ** 2).mean()))

def sagr(y_true, y_pred):
    return float((np.sign(y_true) == np.sign(y_pred)).mean())

rows = []
for j, dim in enumerate(['Valence', 'Arousal']):
    yt, yp = true_va_c[:, j], pred_va_c[:, j]
    rows.append((dim, ccc(yt, yp), pcc(yt, yp), rmse(yt, yp), sagr(yt, yp)))

print('| Dimensi | CCC   | PCC   | RMSE  | SAGR  |')
print('|---------|-------|-------|-------|-------|')
for dim, c_, p_, r_, s_ in rows:
    print('| ' + dim.ljust(7) + ' | ' + format(c_, '.3f') + ' | ' + format(p_, '.3f')
          + ' | ' + format(r_, '.3f') + ' | ' + format(s_, '.3f') + ' |')
print()
print('Mean CCC :', round((rows[0][1] + rows[1][1]) / 2, 4))
print()
print('Referensi silang dari evaluation/va_metrics.py:')
print(json.dumps(evaluate_va(pred_va_c, true_va_c, true_mood_c), indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
panels = [(axes[0], true_va_c, 'Label sebenarnya'), (axes[1], pred_va_c, 'Prediksi model')]
for ax, data, title in panels:
    ax.scatter(data[:, 0], data[:, 1], c=true_mood_c, cmap='tab10',
               s=18, alpha=0.75, vmin=0, vmax=len(MOOD_LABELS) - 1)
    ax.add_patch(plt.Circle((0, 0), NEUTRAL_RADIUS, fill=False,
                            linestyle='--', color='gray', linewidth=1.4))
    ax.axhline(0, color='black', linewidth=0.7)
    ax.axvline(0, color='black', linewidth=0.7)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel('Valence')
    ax.set_ylabel('Arousal')
    ax.set_title(title)
    ax.set_aspect('equal')

handles = []
for i, lab in enumerate(MOOD_LABELS):
    handles.append(plt.Line2D([0], [0], marker='o', linestyle='', markersize=7,
                              color=plt.cm.tab10(i / max(len(MOOD_LABELS) - 1, 1)), label=lab))
axes[1].legend(handles=handles, loc='upper left', fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.savefig('/kaggle/working/va_scatter.png', dpi=180)
print('Disimpan: /kaggle/working/va_scatter.png')
plt.show()

spread_t = true_va_c.std(axis=0)
spread_p = pred_va_c.std(axis=0)
print('Std label    : V', round(float(spread_t[0]), 3), '| A', round(float(spread_t[1]), 3))
print('Std prediksi : V', round(float(spread_p[0]), 3), '| A', round(float(spread_p[1]), 3))
if (spread_p < 0.35 * spread_t).any():
    print('PERINGATAN: prediksi menciut ke rata-rata (regression to the mean).')
    print('Gejala klasik CCCLoss yang terlalu didominasi suku varians. Naikkan beta di DualLoss.')

## 12. Uji falsifikasi - baseline tanpa struktur graf

Inti klaim proposal MoodIn adalah bahwa **struktur graf wajah** membawa informasi yang tidak dimiliki koordinat mentah. Klaim itu harus bisa digugurkan.

Baseline: landmark yang sama, dipipihkan jadi vektor, diberi ke SVM dan MLP - tanpa adjacency, tanpa konvolusi temporal.

**Aturan interpretasi:** kalau ST-GCN tidak mengungguli baseline terbaik pada macro-F1 dengan selisih bermakna, justifikasi arsitektur graf runtuh dan Anda harus menuliskannya apa adanya di Bab IV. Penguji GEMASTIK hampir pasti menanyakan pembanding ini; lebih baik Anda yang menyajikannya lebih dulu.

In [ ]:
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier

def clip_features(clip):
    """Ringkas klip (C, T, V) jadi vektor mean + std + delta -> 408 dimensi."""
    arr = np.asarray(clip)
    mean = arr.mean(axis=1)
    std = arr.std(axis=1)
    delta = np.abs(np.diff(arr, axis=1)).mean(axis=1)
    return np.concatenate([mean.ravel(), std.ravel(), delta.ravel()])

print('Mengekstrak fitur baseline...')
X = np.stack([clip_features(full_ds[i][0]) for i in range(n)])
Y = np.array([int(full_ds[i][2]) for i in range(n)])
print('Bentuk fitur :', X.shape)

baselines = dict()
baselines['Dummy (kelas mayoritas)'] = DummyClassifier(strategy='most_frequent')
baselines['SVM RBF'] = make_pipeline(
    StandardScaler(), SVC(C=10.0, gamma='scale', class_weight='balanced'))
baselines['MLP 256'] = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(256,), max_iter=600,
                  random_state=SEED, early_stopping=True))

results = dict()
for name, clf in baselines.items():
    oof = np.full(n, -1, dtype=np.int64)
    for tr, te in folds:
        clf.fit(X[tr], Y[tr])
        oof[te] = clf.predict(X[te])
    mask = oof >= 0
    results[name] = dict(
        accuracy=float(accuracy_score(Y[mask], oof[mask])),
        macro_f1=float(f1_score(Y[mask], oof[mask], average='macro', zero_division=0)),
    )
    print(name.ljust(26), 'acc', round(results[name]['accuracy'], 4),
          '| macroF1', round(results[name]['macro_f1'], 4))

results['ST-GCN + Transformer'] = dict(accuracy=float(acc), macro_f1=float(macro_f1))

print()
print('| Model | Accuracy | macro-F1 |')
print('|---|---|---|')
for name, m in results.items():
    print('| ' + name + ' | ' + format(m['accuracy'], '.4f') + ' | ' + format(m['macro_f1'], '.4f') + ' |')

best_base = max(m['macro_f1'] for k, m in results.items() if k != 'ST-GCN + Transformer')
margin = macro_f1 - best_base
print()
print('Selisih macro-F1 ST-GCN vs baseline terbaik:', round(margin, 4))
if margin <= 0:
    print('KESIMPULAN: struktur graf BELUM terbukti memberi keuntungan pada dataset ini.')
    print('Tulis apa adanya. Jangan menaikkan angka. Kemungkinan penyebab: dataset terlalu kecil,')
    print('atau CK+ statis sehingga cabang temporal tidak punya sinyal untuk dipelajari.')
elif margin < 0.03:
    print('KESIMPULAN: keunggulan marginal. Butuh uji signifikansi sebelum diklaim.')
else:
    print('KESIMPULAN: struktur graf memberi keuntungan terukur.')

with open('/kaggle/working/baseline_comparison.json', 'w') as fh:
    json.dump(results, fh, indent=2)

## 13. Uji performansi runtime

Tiga angka yang harus masuk Tabel Kinerja proposal:

1. **Latensi inferensi per jendela** (150 frame) - p50 dan p95, bukan rata-rata. p95 yang menentukan apakah UI terasa tersendat.
2. **Throughput** - jendela per detik yang bisa dipertahankan pipeline.
3. **Jejak memori** - peak VRAM/RAM saat inferensi.

> Angka dari notebook ini adalah angka **Kaggle T4**, bukan Jetson. Untuk Tabel 3 proposal, ulangi sel ini di Jetson Orin Nano Super setelah ekspor TensorRT INT8, sambil menjalankan `tegrastats` untuk daya. Sebelum diukur di Jetson, isi tabel dengan `Belum dilakukan`.

In [ ]:
import time, statistics, gc

def benchmark(net, device, batch_size=1, warmup=15, iters=120):
    net = net.to(device).eval()
    sample = torch.randn(batch_size, 2, CLIP_LEN, 68, device=device)
    use_cuda = device.type == 'cuda'

    with torch.no_grad():
        for _ in range(warmup):
            net(sample)
    if use_cuda:
        torch.cuda.synchronize()

    timings = []
    with torch.no_grad():
        for _ in range(iters):
            t0 = time.perf_counter()
            net(sample)
            if use_cuda:
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - t0) * 1000.0)

    timings.sort()
    mean_ms = statistics.mean(timings)
    return dict(
        batch=batch_size,
        device=str(device),
        p50_ms=round(timings[len(timings) // 2], 3),
        p95_ms=round(timings[int(len(timings) * 0.95)], 3),
        p99_ms=round(timings[min(int(len(timings) * 0.99), len(timings) - 1)], 3),
        mean_ms=round(mean_ms, 3),
        std_ms=round(statistics.pstdev(timings), 3),
        windows_per_s=round(1000.0 * batch_size / mean_ms, 2),
    )

ckpt = torch.load(available[0], map_location='cpu')
bench_net = build_model(ckpt['config'])
bench_net.load_state_dict(ckpt['model'])

perf = [benchmark(bench_net, torch.device('cpu'), batch_size=1, warmup=5, iters=40)]
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    perf.append(benchmark(bench_net, torch.device('cuda'), batch_size=1))
    perf.append(benchmark(bench_net, torch.device('cuda'), batch_size=8))
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
else:
    peak_mb = None

print('| Perangkat | Batch | p50 (ms) | p95 (ms) | p99 (ms) | jendela/s |')
print('|---|---|---|---|---|---|')
for r in perf:
    print('| ' + r['device'] + ' | ' + str(r['batch']) + ' | ' + str(r['p50_ms'])
          + ' | ' + str(r['p95_ms']) + ' | ' + str(r['p99_ms'])
          + ' | ' + str(r['windows_per_s']) + ' |')

n_par = sum(p.numel() for p in bench_net.parameters())
print()
print('Parameter   :', format(n_par, ','))
print('Ukuran FP32 :', round(n_par * 4 / 1024**2, 2), 'MB')
print('Peak VRAM   :', 'n/a' if peak_mb is None else round(peak_mb, 2), 'MB')

In [ ]:
# Anggaran waktu: satu jendela 150 frame @30 fps = 5000 ms wall-clock.
budget_ms = CLIP_LEN / 30.0 * 1000.0
gpu_p95 = None
for r in perf:
    if 'cuda' in r['device'] and r['batch'] == 1:
        gpu_p95 = r['p95_ms']

if gpu_p95 is not None:
    print('Anggaran jendela :', budget_ms, 'ms')
    print('p95 terpakai     :', round(gpu_p95 / budget_ms * 100, 3), '%')
    print('Sisa untuk MediaPipe, rPPG, fusi, render UI:', round(budget_ms - gpu_p95, 1), 'ms')

runtime_report = dict(
    measurements=perf,
    params=int(n_par),
    fp32_mb=round(n_par * 4 / 1024**2, 3),
    peak_vram_mb=None if peak_mb is None else round(peak_mb, 2),
    note='Diukur di Kaggle GPU, BUKAN Jetson Orin Nano Super.',
)
with open('/kaggle/working/runtime_performance.json', 'w') as fh:
    json.dump(runtime_report, fh, indent=2)
print('Disimpan: /kaggle/working/runtime_performance.json')

del bench_net
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 14. Ekspor artefak untuk deployment

| Artefak | Untuk apa |
|---|---|
| `stgcn_va.pt` (TorchScript) | jalan di Jetson tanpa perlu mengimpor kode repo |
| `stgcn_va.onnx` | pintu masuk ke `trtexec` untuk engine TensorRT INT8 |
| `stgcn_va_model_card.json` | dibaca connector UI untuk versi model dan metrik |

Di Jetson nanti: `trtexec --onnx=stgcn_va.onnx --int8 --saveEngine=stgcn_va_int8.plan`. Kalibrasi INT8 butuh sekitar 200 klip representatif; tanpa itu akurasi bisa turun tajam.

In [ ]:
EXPORT_DIR = '/kaggle/working/export'
os.makedirs(EXPORT_DIR, exist_ok=True)

ckpt = torch.load(available[0], map_location='cpu')
exp_net = build_model(ckpt['config'])
exp_net.load_state_dict(ckpt['model'])
exp_net.eval()

class VAWrapper(torch.nn.Module):
    """Membungkus output dict menjadi tensor (N, 2) supaya bisa di-trace."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x):
        return self.net(x)['va']

wrapped = VAWrapper(exp_net).eval()
dummy = torch.randn(1, 2, CLIP_LEN, 68)

ts_path = os.path.join(EXPORT_DIR, 'stgcn_va.pt')
torch.jit.save(torch.jit.trace(wrapped, dummy), ts_path)
print('TorchScript :', ts_path, round(os.path.getsize(ts_path) / 1024**2, 2), 'MB')

onnx_path = os.path.join(EXPORT_DIR, 'stgcn_va.onnx')
torch.onnx.export(
    wrapped, dummy, onnx_path,
    input_names=['landmarks'], output_names=['va'],
    dynamic_axes=dict(landmarks=dict([(0, 'batch')]), va=dict([(0, 'batch')])),
    opset_version=17, do_constant_folding=True,
)
print('ONNX        :', onnx_path, round(os.path.getsize(onnx_path) / 1024**2, 2), 'MB')

import onnx
import onnxruntime as ort

onnx.checker.check_model(onnx.load(onnx_path))
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
onnx_out = sess.run(None, dict(landmarks=dummy.numpy()))[0]
with torch.no_grad():
    torch_out = wrapped(dummy).numpy()
max_diff = float(np.abs(onnx_out - torch_out).max())
print('Selisih maks PyTorch vs ONNX:', max_diff)
assert max_diff < 1e-4, 'Ekspor ONNX menyimpang - jangan dipakai untuk TensorRT'
print('Paritas numerik ONNX LULUS')

In [ ]:
MODEL_VERSION = '0.2.0-ckplus-oof'

model_card = dict(
    model_name='moodin-stgcn-graph-transformer-va',
    version=MODEL_VERSION,
    branch='expression',
    contract_version='1.0.0',
    input=dict(
        name='landmarks',
        shape=[1, 2, CLIP_LEN, 68],
        layout='N,C,T,V',
        dtype='float32',
        landmark_scheme='iBUG-68, dipetakan dari MediaPipe FaceMesh 468 via MEDIAPIPE_TO_IBUG68',
        normalization='terpusat di landmark 33, diskalakan jarak antar-mata',
    ),
    output=dict(
        name='va',
        shape=[1, 2],
        order=['valence', 'arousal'],
        value_range=[-1.0, 1.0],
    ),
    mood_mapping=dict(
        function='common.labels.va_to_mood',
        labels=list(MOOD_LABELS),
        neutral_radius=float(NEUTRAL_RADIUS),
    ),
    training=dict(
        dataset='CK+',
        split='subject-independent StratifiedGroupKFold',
        n_folds=N_FOLDS,
        seed=SEED,
        clip_len=CLIP_LEN,
        folds_trained=len(available),
    ),
    metrics=dict(
        accuracy=round(float(acc), 4),
        balanced_accuracy=round(float(bal_acc), 4),
        macro_f1=round(float(macro_f1), 4),
        cohen_kappa=round(float(kappa), 4),
        ccc_valence=round(rows[0][1], 4),
        ccc_arousal=round(rows[1][1], 4),
        rmse_valence=round(rows[0][3], 4),
        rmse_arousal=round(rows[1][3], 4),
        sagr_valence=round(rows[0][4], 4),
        sagr_arousal=round(rows[1][4], 4),
    ),
    baseline_comparison=results,
    runtime=dict(measured_on='Kaggle GPU', jetson_int8='Belum dilakukan'),
    limitations=[
        'CK+ adalah ekspresi berakting di laboratorium, bukan ekspresi spontan.',
        'Label VA diturunkan dari kategori emosi, bukan anotasi VA asli. Aff-Wild2 diperlukan untuk VA sejati.',
        'Belum divalidasi lintas warna kulit (skala Fitzpatrick) maupun kondisi cahaya di balik kaca dua arah.',
    ],
)

card_path = os.path.join(EXPORT_DIR, 'stgcn_va_model_card.json')
with open(card_path, 'w') as fh:
    json.dump(model_card, fh, indent=2, ensure_ascii=False)
print(json.dumps(model_card, indent=2, ensure_ascii=False))

## 15. Kontrak connector ke UI (repo Interface-Display)

Tujuan bagian ini: rekan Anda yang membuat UI **tidak perlu menunggu model selesai dilatih**. Dua berkas yang dihasilkan bisa langsung dipakai paralel:

| Berkas | Untuk siapa |
|---|---|
| `ipc_contract_v1.json` | sumber kebenaran skema payload MQTT, dipakai UI untuk validasi |
| `mock_inference_publisher.py` | publisher palsu yang mengeluarkan payload sah tiap 5 detik |

Dengan mock publisher berjalan, UI bisa dikembangkan penuh tanpa Jetson, tanpa kamera, dan tanpa model.

**Invarian privasi yang tidak boleh dilanggar:** payload hanya berisi skalar numerik. Tidak ada citra wajah, koordinat landmark mentah, maupun embedding identitas yang meninggalkan perangkat. Ini bukan preferensi desain melainkan konsekuensi UU PDP No. 27/2022 yang menggolongkan data biometrik sebagai data pribadi spesifik.

In [ ]:
IPC_VERSION = '1.0.0'

topic_presence = dict(
    qos=1, retained=True, lwt=True, direction='engine -> ui',
    payload=dict(state='online atau offline', device_id='str', ts='float epoch'),
)

topic_quality = dict(
    qos=0, retained=False, direction='engine -> ui',
    payload=dict(
        face_detected='bool',
        landmark_confidence='float 0..1',
        roi_snr_db='float',
        illuminance_ok='bool',
        buffer_fill='float 0..1, progres pengisian jendela 150 frame',
        ts='float epoch',
    ),
)

topic_result = dict(
    qos=1, retained=False, direction='engine -> ui',
    payload=dict(
        mood_id='int 0..4',
        mood_label='str, salah satu dari MOOD_LABELS',
        confidence='float 0..1',
        valence='float -1..1',
        arousal='float -1..1',
        heart_rate_bpm='float atau null',
        hrv_ms='float atau null',
        respiration_bpm='float atau null',
        physio_used='bool, false berarti fusi jatuh ke ekspresi saja',
        model_version='str',
        window_start_ts='float epoch',
        ts='float epoch',
    ),
)

topic_command = dict(
    qos=1, retained=False, direction='ui -> engine',
    payload=dict(
        command='start, stop, recalibrate, enroll_face, logout, set_privacy_mode',
        args='object opsional',
        ts='float epoch',
    ),
)

topics = dict()
topics['moodin/presence'] = topic_presence
topics['moodin/signal/quality'] = topic_quality
topics['moodin/inference/result'] = topic_result
topics['moodin/ui/command'] = topic_command

ipc_contract = dict(
    contract_version=IPC_VERSION,
    transport='MQTT',
    broker=dict(dev='localhost:1883', prod='tls://HOST:8883'),
    topics=topics,
    mood_labels=list(MOOD_LABELS),
    privacy_invariant=(
        'Tidak ada citra wajah, koordinat landmark mentah, maupun embedding identitas '
        'yang boleh dipublikasikan ke topik mana pun. Hanya skalar numerik.'
    ),
    degradation_rules=[
        'physio_used=false -> UI menampilkan mood tanpa kartu tanda vital.',
        'face_detected=false selama lebih dari 3 detik -> UI kembali ke layar idle.',
        'buffer_fill kurang dari 1.0 -> UI menampilkan progress ring, jangan tampilkan mood basi.',
        'presence=offline lewat LWT -> UI menampilkan banner engine terputus.',
    ],
)

contract_path = os.path.join(EXPORT_DIR, 'ipc_contract_v1.json')
with open(contract_path, 'w') as fh:
    json.dump(ipc_contract, fh, indent=2, ensure_ascii=False)
print('Kontrak IPC disimpan:', contract_path)
print(json.dumps(topic_result, indent=2, ensure_ascii=False))

In [ ]:
%%writefile /kaggle/working/export/mock_inference_publisher.py
"""Mock MoodIn inference publisher - kontrak IPC v1.0.0.

Dipakai tim Interface-Display untuk mengembangkan UI tanpa Jetson, kamera, atau model.

    pip install paho-mqtt==2.1.0
    mosquitto -p 1883 &
    python mock_inference_publisher.py --interval 5 --scenario cycle
"""

import argparse
import json
import math
import random
import time

import paho.mqtt.client as mqtt

CONTRACT_VERSION = "1.0.0"
MODEL_VERSION = "0.2.0-mock"
MOOD_LABELS = ["Ceria", "Rileks", "Cemas", "Tertekan", "Netral"]
NEUTRAL_RADIUS = 0.2

TOPIC_PRESENCE = "moodin/presence"
TOPIC_QUALITY = "moodin/signal/quality"
TOPIC_RESULT = "moodin/inference/result"
TOPIC_COMMAND = "moodin/ui/command"

SCENARIOS = dict(
    cycle=[(0.6, 0.5), (0.5, -0.4), (-0.5, 0.6), (-0.6, -0.5), (0.05, 0.05)],
    stable=[(0.4, 0.3)],
    degraded=[(0.3, 0.2)],
    noisy=None,
)


def va_to_mood(valence, arousal):
    """Kuantisasi circumplex Russell. Harus identik dengan common.labels.va_to_mood."""
    if math.hypot(valence, arousal) < NEUTRAL_RADIUS:
        return 4, MOOD_LABELS[4]
    if valence >= 0 and arousal >= 0:
        return 0, MOOD_LABELS[0]
    if valence >= 0 and arousal < 0:
        return 1, MOOD_LABELS[1]
    if valence < 0 and arousal >= 0:
        return 2, MOOD_LABELS[2]
    return 3, MOOD_LABELS[3]


def clamp(x):
    return max(-1.0, min(1.0, x))


def next_va(scenario, step):
    if scenario == "noisy":
        return random.uniform(-1.0, 1.0), random.uniform(-1.0, 1.0)
    seq = SCENARIOS[scenario]
    base_v, base_a = seq[step % len(seq)]
    return clamp(base_v + random.gauss(0.0, 0.06)), clamp(base_a + random.gauss(0.0, 0.06))

In [ ]:
%%writefile -a /kaggle/working/export/mock_inference_publisher.py


def build_result(valence, arousal, physio_used, window_start):
    mood_id, mood_label = va_to_mood(valence, arousal)
    payload = dict(
        mood_id=mood_id,
        mood_label=mood_label,
        confidence=round(random.uniform(0.62, 0.94), 3),
        valence=round(valence, 4),
        arousal=round(arousal, 4),
        heart_rate_bpm=round(random.uniform(62.0, 88.0), 1) if physio_used else None,
        hrv_ms=round(random.uniform(28.0, 74.0), 1) if physio_used else None,
        respiration_bpm=round(random.uniform(12.0, 18.0), 1) if physio_used else None,
        physio_used=physio_used,
        model_version=MODEL_VERSION,
        contract_version=CONTRACT_VERSION,
        window_start_ts=round(window_start, 3),
        ts=round(time.time(), 3),
    )
    return payload


def build_quality(buffer_fill, face_detected):
    return dict(
        face_detected=face_detected,
        landmark_confidence=round(random.uniform(0.80, 0.99), 3) if face_detected else 0.0,
        roi_snr_db=round(random.uniform(4.5, 12.0), 2) if face_detected else 0.0,
        illuminance_ok=True,
        buffer_fill=round(buffer_fill, 3),
        ts=round(time.time(), 3),
    )


def on_command(client, userdata, msg):
    try:
        cmd = json.loads(msg.payload.decode())
    except json.JSONDecodeError:
        print("[cmd] payload bukan JSON:", msg.payload[:80])
        return
    print("[cmd] diterima:", cmd.get("command"), cmd.get("args"))


def main():
    parser = argparse.ArgumentParser(description="Mock MoodIn inference publisher")
    parser.add_argument("--host", default="localhost")
    parser.add_argument("--port", type=int, default=1883)
    parser.add_argument("--interval", type=float, default=5.0,
                        help="detik antar hasil inferensi; 5.0 = jendela 150 frame @30 fps")
    parser.add_argument("--scenario", default="cycle", choices=sorted(SCENARIOS))
    parser.add_argument("--device-id", default="moodin-mirror-01")
    args = parser.parse_args()

    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="moodin-mock-engine")
    offline = json.dumps(dict(state="offline", device_id=args.device_id, ts=time.time()))
    client.will_set(TOPIC_PRESENCE, offline, qos=1, retain=True)
    client.on_message = on_command
    client.connect(args.host, args.port, keepalive=30)
    client.subscribe(TOPIC_COMMAND, qos=1)
    client.loop_start()

    online = json.dumps(dict(state="online", device_id=args.device_id, ts=time.time()))
    client.publish(TOPIC_PRESENCE, online, qos=1, retain=True)
    print("engine mock online di", args.host, args.port, "| skenario", args.scenario)

    step = 0
    try:
        while True:
            window_start = time.time()
            ticks = max(int(args.interval / 0.5), 1)
            for i in range(ticks):
                fill = (i + 1) / ticks
                client.publish(TOPIC_QUALITY, json.dumps(build_quality(fill, True)), qos=0)
                time.sleep(args.interval / ticks)

            valence, arousal = next_va(args.scenario, step)
            physio_used = args.scenario != "degraded"
            payload = build_result(valence, arousal, physio_used, window_start)
            client.publish(TOPIC_RESULT, json.dumps(payload), qos=1)
            print("[result]", payload["mood_label"],
                  "V=" + str(payload["valence"]),
                  "A=" + str(payload["arousal"]),
                  "physio=" + str(physio_used))
            step += 1
    except KeyboardInterrupt:
        print("\nberhenti")
    finally:
        client.publish(TOPIC_PRESENCE, offline, qos=1, retain=True)
        client.loop_stop()
        client.disconnect()


if __name__ == "__main__":
    main()

## 16. Ringkasan hasil dan langkah berikutnya

Sel di bawah mengumpulkan seluruh artefak ke satu berkas ZIP untuk diserahkan ke repo.

### Yang diserahkan ke Interface-Display

| Berkas | Peran |
|---|---|
| `ipc_contract_v1.json` | sumber kebenaran skema payload |
| `mock_inference_publisher.py` | jalankan sekarang, UI bisa dikembangkan paralel |
| `stgcn_va_model_card.json` | versi model dan metrik yang ditampilkan di layar About |

### Yang diserahkan ke Software-Architecture

| Berkas | Peran |
|---|---|
| `stgcn_va.pt`, `stgcn_va.onnx` | artefak deployment |
| `oof_predictions.npz` | prediksi out-of-fold untuk kalibrasi bobot fusi w1, w2 |
| `runtime_performance.json`, `baseline_comparison.json` | isi Tabel Kinerja proposal |
| `confusion_matrix.png`, `va_scatter.png` | gambar Bab IV |

### Urutan kerja berikutnya

1. **Verifikasi peta landmark** dengan LCK+ sebelum melatih apa pun lagi. Kalau `MEDIAPIPE_TO_IBUG68` salah, semua angka di atas tidak bermakna.
2. **Jalankan CV 10 fold penuh** (`RUN_FULL_CV = True`) dan laporkan mean +/- std, bukan angka satu fold.
3. **Ukur ulang di Jetson** setelah `trtexec --int8`, sambil `tegrastats` untuk daya. Sebelum itu, Tabel 3 tetap `Belum dilakukan`.
4. **Kalibrasi bobot fusi** `w1`, `w2` di `LateFusionVABridge` memakai `oof_predictions.npz` plus keluaran rPPG pada rekaman yang sama.
5. **Ganti ke Aff-Wild2** setelah EULA disetujui. CK+ hanya baseline; label VA-nya turunan, bukan anotasi asli.

In [ ]:
import shutil, glob

for src in ['/kaggle/working/confusion_matrix.png',
            '/kaggle/working/va_scatter.png',
            '/kaggle/working/oof_predictions.npz',
            '/kaggle/working/runtime_performance.json',
            '/kaggle/working/baseline_comparison.json',
            FOLD_PATH]:
    if os.path.exists(src):
        shutil.copy2(src, EXPORT_DIR)

bundle = shutil.make_archive('/kaggle/working/moodin_stgcn_artifacts', 'zip', EXPORT_DIR)
print('Bundle :', bundle, round(os.path.getsize(bundle) / 1024, 1), 'KB')
print()
for path in sorted(glob.glob(os.path.join(EXPORT_DIR, '*'))):
    print(' ', os.path.basename(path).ljust(34), round(os.path.getsize(path) / 1024, 1), 'KB')
print()
print('Unduh dari panel Output Kaggle, lalu commit ke repo terkait.')